<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_89_constitutional_ai_rlaif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⚖️ **Module 89 — Constitutional AI / RLAIF Deep Dive** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 12 · Production Extensions


# Module 89 — Constitutional AI / RLAIF Deep Dive

> RLHF needed tens of thousands of human preference pairs. **Constitutional AI** (Bai et al., Anthropic 2022) replaced most of those humans with **the model itself**, guided by a *written constitution*. The result — **RLAIF** (RL from AI Feedback) — is now the alignment workhorse behind Claude, Llama-3-Instruct, Qwen2-Chat, Nemotron, and every open chat fine-tune in 2026. This module is the production deep-dive: the **SL-CAI → RL-CAI** pipeline, the **principle catalog**, **direct preference optimization (DPO)** vs PPO, **judge-model selection**, **mode-collapse + jailbreak defenses**, **Claude's Sparrow-style critique-and-revise loop**, and modern variants — **IPO · KTO · ORPO · SimPO · RLHF-V · SPIN · DRPO**.

### What you'll cover
1. RLHF recap → why RLAIF beats it on cost / scale
2. **Constitutional AI** — the principle catalog + critique-and-revise loop
3. **SL-CAI** (supervised stage) — generate, critique, revise, SFT
4. **RL-CAI** (RL stage) — pairwise judge → preference dataset → DPO / PPO
5. **DPO** math + why it replaced PPO almost everywhere
6. **The reward model** — when you need one, when you don't
7. The modern variant zoo — **IPO · KTO · ORPO · SimPO · DRPO · SPIN**
8. **Judge model selection** — when GPT-4o, Claude, Gemini, or Llama-Guard wins
9. **Safety stack** — refusal · over-refusal · jailbreaks · evals
10. **Production pipeline** in 2026 — distilabel · TRL · Skywork · Llama-Guard · Argilla


## 1 · From RLHF to RLAIF — why we changed engines

**RLHF (Christiano 2017 → InstructGPT 2022)** had three stages:
1. SFT on demonstrations
2. **Train a reward model** on ~50-100k human preference pairs (`A > B`)
3. **PPO** the SFT model against that reward

It works — but the human-preference stage is the bottleneck:
- ~$2-10 per pair × 100k = $200k-1M per round
- 6-8 weeks per labeling cycle
- Inter-annotator agreement ~70% → noisy reward

**Constitutional AI / RLAIF** swaps the human in stage 2 for an **LLM judge** guided by a *written constitution*. Costs drop 100-1000×, cycle time goes from weeks to hours, and — surprisingly — quality on most rubrics matches or exceeds RLHF (Lee et al. 2023 RLAIF paper; Bai et al. 2022 CAI paper).

```
HUMAN PIPELINE                    AI-FEEDBACK PIPELINE
prompt                            prompt
  │                                 │
generator → A, B                  generator → A, B
  │                                 │
HUMAN labels A > B               LLM judges (constitution) A > B
  │                                 │
reward model R                   reward model R   (or skip with DPO)
  │                                 │
PPO                               PPO / DPO / IPO / KTO ...
```


## 2 · The Constitution — Anthropic's principle catalog

A **constitution** is a written list of principles the judge applies. Anthropic's published one for Claude has ~75 principles, drawn from:
- UN Declaration of Human Rights
- Apple's Terms of Service
- DeepMind's Sparrow rules
- Anthropic-internal rules ("don't help with biothreats", "don't impersonate", ...)

A few representative principles (paraphrased from the public CAI paper):

| # | Principle |
|---|---|
| 1 | "Please choose the response that is most supportive of life, liberty, and personal security." |
| 2 | "Choose the response that is most age-appropriate for children." |
| 3 | "Choose the response that is least likely to be viewed as harmful or offensive to a non-Western audience." |
| 4 | "Choose the response that is more thoughtful, respectful, and ethically grounded." |
| 5 | "Choose the response that least implies that the AI has a personal identity." |
| 6 | "Choose the response least likely to mislead or deceive." |

At judge time, **one principle is sampled per comparison**. This is critical: sampling one principle per (A, B) pair means the judge can't optimize for any single rubric — the model learns the *intersection* of principles.

> 📜 **Why a constitution at all?** Two reasons: (1) Auditability — "why did the model refuse?" has a written answer. (2) Iterability — changing one line in the constitution propagates through every future training run. RLHF preferences are opaque and frozen the moment they're collected.


## 3 · SL-CAI — the supervised stage

Stage 1 of CAI is purely SFT (no RL). It works in a 4-step loop applied to a **red-team prompt set** (prompts likely to elicit harmful behavior):

```
prompt p (e.g. "How do I hotwire a car?")
   │
   ▼
1. GENERATE     →  initial response r₀  (from helpful-only model)
   │
   ▼
2. CRITIQUE     →  prompt LLM with a constitution principle:
                   "Identify ways in which r₀ is harmful or unethical."
   │              produces critique c
   ▼
3. REVISE       →  prompt LLM: "Rewrite r₀ to address critique c."
                   produces revised response r₁
   │
   ▼
4. SFT          →  fine-tune the model on (p, r₁) pairs
```

Run the loop 1-4 times per prompt. The output dataset is `{(prompt, final_revision)}` — pure SFT pairs. **No human labels.** The trained model is the **SL-CAI model** and is already much safer than the helpful-only base.

Why this works: red-team prompts make the helpful-only model misbehave; the constitution-guided critique makes the same model articulate *why* the misbehavior is wrong; the revision step internalizes the fix. The model is teaching itself with the constitution as the curriculum.


In [ ]:
# SL-CAI critique-and-revise loop (sketch, vendor-agnostic)
PRINCIPLE_POOL = [
    "Identify any harm, dishonesty, toxicity, or illegality in the response above.",
    "Identify ways the response is racist, sexist, or otherwise discriminatory.",
    "Identify ways the response could be unsafe for a young person.",
    # ...75 more
]

def sl_cai_one(prompt, llm, rounds=2):
    import random
    r = llm.complete(prompt)                              # r0 — initial answer
    for _ in range(rounds):
        principle = random.choice(PRINCIPLE_POOL)
        critique  = llm.complete(
            f"Response:\n{r}\n\nTask: {principle}\nCritique:")
        r = llm.complete(
            f"Original response:\n{r}\n\nCritique:\n{critique}\n\n"
            "Rewrite the response to address the critique while remaining helpful:")
    return prompt, r                                      # final SFT pair


## 4 · RL-CAI — the RL stage with an LLM judge

After SL-CAI, the second stage replaces RLHF's human labelers with the LLM judge. For each prompt the SL-CAI model generates **two candidates** `A` and `B`. The judge picks the winner under a *sampled* constitution principle.

```
1. Generate A, B from current policy
2. Sample principle 'p' from constitution
3. Judge prompt:
      "Consider these two responses. Which better follows this principle?
       Principle: {p}
       Response A: ...
       Response B: ...
       Answer with a single letter, A or B."
4. Record (prompt, A, B, label) — produces RLAIF preference dataset
5. Train reward model R on those pairs  (optional with DPO/IPO/etc.)
6. PPO / DPO the policy against R
```

The output is a fully synthetic preference dataset of ~100k-1M pairs. Stages 5-6 are identical to RLHF — only the labeler changed.

> 🧠 **Chain-of-thought judge.** Bai et al. 2022 found that asking the judge to **first write a paragraph of reasoning** before answering A/B raised judge accuracy by 5-15 points. Modern judges (GPT-4o, Claude-3.7, Llama-Guard-3, Skywork-Reward, ArmoRM) all use CoT internally.


## 5 · DPO — why PPO got benched

**PPO** (used in InstructGPT) is heavy: it requires a separate reward model, a value head, a frozen reference policy, KL clipping, and careful entropy bonuses. **Direct Preference Optimization** (Rafailov et al. NeurIPS 2023) showed you can skip all of that.

**The DPO trick.** Given preference data `(prompt x, winner y_w, loser y_l)`, DPO directly trains the policy with the loss:

```
L_DPO = −E[ log σ( β · ( log π_θ(y_w|x)/π_ref(y_w|x) − log π_θ(y_l|x)/π_ref(y_l|x) ) ) ]
```

Where:
- `π_θ` = policy being trained
- `π_ref` = frozen reference (usually the SFT model)
- `β` ≈ 0.01-0.5 controls how far the policy can drift from `π_ref`
- `σ` = sigmoid

In plain English: **push up the probability of the winner relative to the reference, push down the probability of the loser relative to the reference, by a margin controlled by β.**

| | PPO | DPO |
|---|---|---|
| Needs reward model | yes | **no** |
| Needs value head | yes | **no** |
| Optimizer pass per step | 4 (policy, value, reward, ref) | **2 (policy, ref)** |
| Hyperparam pain | high (KL coeff, clip, value coeff, GAE) | **low (just β)** |
| Stability | finicky | **stable** |
| Open recipe | InstructGPT | **Zephyr-7B-β, Llama-3-Instruct, Qwen2.5-Chat, Mistral-Instruct, Gemma-2-IT all use DPO or a DPO variant.** |

By 2026 PPO is reserved for cases where you really do want a reusable reward model (production RLHF-V, agentic systems, online learning).


In [ ]:
# DPO loss in 12 lines (TRL-style)
import torch, torch.nn.functional as F

def dpo_loss(policy, ref, batch, beta=0.1):
    # batch has prompt, chosen, rejected token ids
    logp_pol_w = log_prob(policy, batch.prompt, batch.chosen)
    logp_pol_l = log_prob(policy, batch.prompt, batch.rejected)
    with torch.no_grad():
        logp_ref_w = log_prob(ref, batch.prompt, batch.chosen)
        logp_ref_l = log_prob(ref, batch.prompt, batch.rejected)
    logits = beta * ((logp_pol_w - logp_ref_w) - (logp_pol_l - logp_ref_l))
    # equivalent to BCE with a target of 1
    return -F.logsigmoid(logits).mean()


## 6 · When you still need a reward model

Three cases where DPO's "no reward model" claim breaks down:

1. **Online RL.** DPO is offline — it trains on a fixed preference dataset. If you want the model to keep learning as users react in production, you need a callable scalar `R(prompt, response)` — i.e. a reward model.
2. **Best-of-N sampling.** Cheap inference-time alignment: sample 8-64 responses, score each with `R`, return the top-1. No fine-tuning needed.
3. **Process reward.** For long-horizon reasoning, you score **each step** of a chain of thought (PRM, process reward model). Used in OpenAI's MathLM, DeepSeek-Math, and o1/R1-style training.

**Best open reward models (May 2026):**

| Model | Size | Notes |
|---|---|---|
| **Nemotron-4-340B-Reward** | 340B MoE | Strongest open; built into NVIDIA's synthetic-data stack |
| **Skywork-Reward-Llama-3.1-8B** | 8B | Best size/quality tradeoff; tops RewardBench |
| **ArmoRM-Llama3-8B** | 8B | Multi-objective (helpful / safe / honest / verbose / ...) |
| **InternLM2-7B-Reward** | 7B | Strong on Chinese/multilingual |
| **Eurus-RM-7B** | 7B | Tuned for reasoning / code |
| **Llama-Guard-3-8B** | 8B | *Safety only* — refusal-classifier, not a quality reward |

> ⚠️ **Reward hacking.** Reward models drift; the policy learns to game them. Standard mitigations: (a) ensemble of rewards, (b) KL penalty to reference, (c) periodic retraining on fresh preferences, (d) **adversarial evals** ("write a response that maximizes R but is bad").


## 7 · The 2026 preference-optimization zoo

Post-DPO research exploded. The most-used variants:

| Method | Year | One-line idea | When to use |
|---|---|---|---|
| **DPO** | 2023 | Direct preference loss vs reference | Default; biggest ecosystem |
| **IPO** (Identity-PO) | 2023 | Replace sigmoid with a *bounded* surrogate; less overfitting | When DPO blows past the reference |
| **KTO** (Kahneman-Tversky) | 2024 | **No pairs needed** — just `(prompt, response, good/bad)` flags | When you have thumbs up/down logs, not pairs |
| **ORPO** | 2024 | Odd-Ratio PO; combines SFT + preference in **one stage**, no reference model | Cheapest pipeline; great for small models |
| **SimPO** | 2024 | DPO without a reference model — uses average log-prob | When you can't keep a ref model in memory |
| **DRPO** / **R-DPO** | 2024 | Robust DPO — down-weights noisy labels | Crowd-sourced labels |
| **SPIN** | 2024 | Self-play: model's own outputs become rejected, GT becomes chosen | Boost SFT model without preference data |
| **NCA** / **CPO** / **RPO** | 2024-25 | Various contrastive tweaks | Niche |
| **GRPO** (DeepSeek) | 2024 | Group-relative PPO with verifier reward — *no* value head | Verifiable tasks (math, code, reasoning) — what R1 used |

**Decision tree:**

```
Have pairs (A>B)?
├── Yes → DPO (default) ── overfitting? → IPO
│                         ── single stage? → ORPO
│                         ── no ref RAM? → SimPO
└── No  → thumbs only?  → KTO
       → only good data → SPIN
       → verifiable?    → GRPO / RFT (M88)
```

> 🎯 **The 2026 consensus.** For chat models, ORPO during SFT or DPO after SFT. For reasoning models, GRPO. For agentic / online, keep a reward model (Skywork or ArmoRM) and use best-of-N or PPO.


## 8 · Judge model selection

The **judge** is the single most important choice in RLAIF — it's literally the gradient signal. Three rules:

1. **The judge must be stronger than the policy.** Distillation only works when teacher > student. A weak judge teaches the policy to game it.
2. **Use multiple judges and aggregate.** Single-judge bias is real. Two independent judges + majority vote raises agreement with humans by ~10 points.
3. **Calibrate length / verbosity bias.** Every LLM judge prefers longer responses. Either control length explicitly in the prompt or add a length penalty in the reward.

| Judge | Strengths | Weaknesses |
|---|---|---|
| **GPT-4o / GPT-4-Turbo** | High agreement w/ humans, JSON mode | Closed; ToS restrictions on competing models |
| **Claude-3.7-Sonnet / Opus** | Best on nuance / safety / reasoning | Closed; rate limits |
| **Gemini-2.5-Pro** | Long context, fast | Vendor lock-in for some workflows |
| **Llama-3.1-70B / 405B-Instruct** | Open, no ToS issues | Length bias, weaker on safety nuance |
| **Nemotron-4-340B-Reward** | Reward-style scalar; fast batching | No CoT explanation |
| **Skywork-Reward-Llama-3.1-8B** | Cheap, fast, strong on RewardBench | 8B knows less of the world |
| **ArmoRM** | Multi-axis (helpful/safe/honest separately) | Heavier to deploy |
| **Llama-Guard-3-8B** | Specialized safety classifier | NOT a quality judge |

**Pattern for production:** Skywork-Reward for quality + Llama-Guard-3 for safety + GPT-4o spot-check for a 1% audit slice.


## 9 · Safety stack — refusals, over-refusal, jailbreaks

Alignment is not just "refuse harmful prompts." A real safety stack has four sub-problems:

| Sub-problem | What it is | Eval |
|---|---|---|
| **Harmful compliance** | Model helps with disallowed task | **AdvBench, HarmBench, JBB-Behaviors** |
| **Over-refusal** | Model refuses benign prompts ("how do I kill a Python process?") | **XSTest, OR-Bench** |
| **Jailbreaks** | Adversarial prompts that bypass training | **PAIR, AutoDAN, GCG, MART** |
| **Sycophancy** | Model agrees with user even when wrong | **Sycophancy-Eval, TruthfulQA** |

**Production safety pattern (Llama-3 / Claude / GPT-4o all use a variant):**

```
                ┌─ system prompt (positive guidance)
prompt ────────┤
                └─ input safety classifier (Llama-Guard-3, prompt-side)
                       │
                       ▼
                policy generates response
                       │
                       ▼
                output safety classifier (Llama-Guard-3, response-side)
                       │
                       ▼
                response or refusal-message
```

**Defense-in-depth idea:** training-time CAI/DPO gets ~95% of harmful prompts; the input/output classifiers catch most of the remaining 5% plus novel jailbreaks; periodic red-teaming feeds new examples back into the next CAI cycle.

> 🧪 **Standard 2026 eval suite.** AdvBench (50 base-rate harms) + XSTest (250 over-refusal) + AlpacaEval-2 (instruction following) + Arena-Hard-Auto (general quality) + MMLU/GSM8K (capability) — track *all five* every checkpoint to catch alignment tax.


## 10 · A 2026 production pipeline — end to end

A complete open-source alignment pipeline for a custom Llama-3-8B or Qwen-2.5-7B base:

```
                ┌─────────────────────────────────────────────────────┐
                │  STAGE 1 — SFT                                      │
                │  ─ Magpie + OpenHermes + domain data                │
                │  ─ TRL SFTTrainer (M88 recipe)                      │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │  STAGE 2 — SL-CAI                                   │
                │  ─ red-team prompt set (HarmBench, AdvBench seeds)  │
                │  ─ critique-and-revise loop (this notebook)         │
                │  ─ SFT on revisions                                 │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │  STAGE 3 — RLAIF preference data                    │
                │  ─ generate A, B pairs (vLLM, batch)                │
                │  ─ judge with Skywork-Reward + Llama-Guard-3        │
                │  ─ filter contradictions / ties                      │
                │  ─ output: ~500k (prompt, chosen, rejected)         │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │  STAGE 4 — DPO (or ORPO / SimPO)                    │
                │  ─ TRL DPOTrainer, β=0.1, 1-3 epochs                │
                │  ─ LoRA optional (PEFT) for cheap experiments       │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │  STAGE 5 — Eval + safety classifiers                │
                │  ─ AlpacaEval-2 + Arena-Hard + XSTest + AdvBench    │
                │  ─ wrap with Llama-Guard-3 input+output             │
                │  ─ red-team → loop back to Stage 2                  │
                └─────────────────────────────────────────────────────┘
```

**Tooling.** `distilabel` (synthetic preferences), `trl` (SFT/DPO/PPO/KTO/ORPO), `vllm` (batch judge), `peft` (LoRA), `argilla` (HITL curation), `lm-eval-harness` + `inspect-ai` (evals).

> 🔮 **What's next.** RLAIF + verifiers is the recipe behind every 2025-2026 reasoning model (o1, R1, Claude-Thinking, Gemini-Thinking). The constitution is increasingly **learned** — Anthropic's "Constitutional Classifiers" and OpenAI's "Deliberative Alignment" both have the model learn what counts as harmful at training time rather than baking it into a fixed list.


In [ ]:
# TRL DPO in 15 lines — the de-facto open recipe
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

base = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(base)
ref   = AutoModelForCausalLM.from_pretrained(base)         # frozen reference
tok   = AutoTokenizer.from_pretrained(base)

ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")

cfg = DPOConfig(
    output_dir="qwen7b-dpo",
    beta=0.1,                      # smaller = stay near ref
    learning_rate=5e-7,            # tiny LR — DPO is sensitive
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    bf16=True,
)
DPOTrainer(model, ref_model=ref, args=cfg, train_dataset=ds, tokenizer=tok).train()


## ✅ Recap

- **RLAIF replaces ~all human labelers with an LLM judge guided by a written constitution** — 100-1000× cheaper, hours instead of weeks.
- **CAI** has two stages: **SL-CAI** (critique-and-revise → SFT) and **RL-CAI** (judge → preference pairs → PPO / DPO).
- **DPO** killed PPO for most chat fine-tunes — one loss, one β, no reward model. Modern variants: **IPO · KTO · ORPO · SimPO · DRPO · SPIN · GRPO**.
- Reward models still matter for **online RL, best-of-N, process reward**. Best 2026 open RMs: **Skywork · ArmoRM · Nemotron-4-Reward**.
- **Judge selection is the single biggest lever** — judge must be stronger than policy; use multiple judges; mind length bias.
- **Safety stack** = training-time CAI/DPO + input classifier + output classifier + red-team loop. Always co-eval **AdvBench + XSTest + AlpacaEval + MMLU** to catch alignment tax.
- 2026 stack: **distilabel + trl + vllm + peft + Skywork + Llama-Guard-3 + argilla + lm-eval-harness**.

Next: **M90 — Edge / On-device AI** (final module, closes the course).
